### Let's preprocess BOBs !!

1. Load the images and their corresponding segmentations as pairs
2. Reorient to RAS (right-anterior-superior), check that all BOBs volumes are RAS
3. Fill empty/NaN values by 0
4. Resample to a common spacing
5. Crop to the brain (remove background)
6. Pad to a fixed shape
7. Normalize intensities: per volume 
8. Remap labels 

PS: this pipeline follows the data transforms from https://github.com/Medical-Image-Analysis-Laboratory/segsynthtemplate/blob/main/configs/data/transforms/inference.yaml

### 1. Load images and their corresponding segmentations as pairs 

In [ ]:
from pathlib import Path 
import sys

PROJECT_ROOT = Path.cwd().parent 
sys.path.append(str(PROJECT_ROOT))

from src.preprocess import (find_image_label_pairs)

DATA_DIR = PROJECT_ROOT / "data" / "bobs" / "resampled75"
# BOBs has two segmentations per scan: we use the bobs9 one (not aseg)
pairs = find_image_label_pairs(DATA_DIR, seg_suffix="_desc-bobs9_dseg")


print("Number of pairs: ", len(pairs))
print("First 5 pairs: ", pairs[:5])

### 2. Reorient to RAS

In [ ]:
import nibabel as nib 

from src.preprocess import reorient_to_ras

n_not_ras = 0
for img_path, seg_path in pairs:
    if nib.aff2axcodes(nib.load(img_path).affine) != ("R", "A", "S"):
        n_not_ras += 1
    img = reorient_to_ras(nib.load(img_path))
    seg = reorient_to_ras(nib.load(seg_path))
    assert nib.aff2axcodes(img.affine) == ("R", "A", "S"), img_path.name
    assert nib.aff2axcodes(seg.affine) == ("R", "A", "S"), seg_path.name

print(n_not_ras, "pairs were not RAS before reorienting")
print("All", len(pairs), "pairs are RAS")

let's check visually that it worked

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pick the first subject that was not RAS (if all were RAS already, just take the first one)
not_ras = [
    (i, s) for i, s in pairs
    if nib.aff2axcodes(nib.load(i).affine) != ("R", "A", "S")
]
img_path, seg_path = not_ras[0] if not_ras else pairs[0]

img_before = nib.load(img_path)
img_after = reorient_to_ras(img_before)
seg_after = reorient_to_ras(nib.load(seg_path))

before = img_before.get_fdata()
after = img_after.get_fdata()
mask = np.asarray(seg_after.dataobj)

z = after.shape[2] // 2   # middle axial slice

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

axes[0].imshow(before[:, :, z].T, cmap="gray", origin="lower")
axes[0].set_title(f"{img_path.name.split('_')[0]} before: {nib.aff2axcodes(img_before.affine)}")

axes[1].imshow(after[:, :, z].T, cmap="gray", origin="lower")
axes[1].contour((mask[:, :, z] > 0).T, levels=[0.5], colors="red", linewidths=0.8)
axes[1].set_title(f"after: {nib.aff2axcodes(img_after.affine)} + mask")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3. Fill empty/NaN values with 0

In [ ]:
from src.preprocess import fill_nan

with_bad_values = 0

for img_path, seg_path in pairs:
    img = reorient_to_ras(nib.load(img_path))
    img_filled = fill_nan(img)

    before = img.get_fdata()
    after = img_filled.get_fdata()

    bad_before = np.isnan(before).sum() + np.isinf(before).sum()
    if bad_before > 0:
        with_bad_values += 1
        print(f"{img_path.name} has {bad_before} bad values before filling.")

    assert not np.isnan(after).any() and not np.isinf(after).any(), img_path.name
    assert nib.aff2axcodes(img_filled.affine) == ("R", "A", "S"), img_path.name

print(f"Done: {len(pairs)} images checked, {with_bad_values} had NaN/inf")

### 4. Resample to common spacing

In [ ]:
from src.preprocess import resample_to_spacing

# TODO: confirm the target spacing with the supervisors (same as FeTA?)
TARGET_SPACING = (0.5, 0.5, 0.5)

img_path, seg_path = pairs[0]
img = fill_nan(reorient_to_ras(nib.load(img_path)))
seg = reorient_to_ras(nib.load(seg_path))
img_r, seg_r = resample_to_spacing(img, seg, spacing=TARGET_SPACING)

print("Spacing before:", img.header.get_zooms()[:3], "after:", img_r.header.get_zooms()[:3])
print("Shape before:  ", img.shape, "after:", img_r.shape, "| mask:", seg_r.shape)
print("Labels before: ", np.unique(np.asarray(seg.dataobj)))
print("Labels after:  ", np.unique(np.asarray(seg_r.dataobj)))
print("Still RAS:", nib.aff2axcodes(img_r.affine))

### 5. Remove background

In [ ]:
import matplotlib.patches as patches

from src.preprocess import crop_to_foreground

img_path, seg_path = pairs[0]
img = fill_nan(reorient_to_ras(nib.load(img_path)))
seg = reorient_to_ras(nib.load(seg_path))
img_r, seg_r = resample_to_spacing(img, seg, spacing=TARGET_SPACING)
img_c, seg_c = crop_to_foreground(img_r, seg_r)

print("Before:", img_r.shape, "After:", img_c.shape, seg_c.shape)

# Where the crop starts in the resampled volume (from the two affines),
# so both panels show the SAME slice
start = np.round(np.linalg.inv(img_r.affine) @ img_c.affine @ [0, 0, 0, 1])[:3].astype(int)
z_c = img_c.shape[2] // 2      # slice in the cropped volume
z = start[2] + z_c             # same slice in the uncropped volume

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(img_r.get_fdata()[:, :, z].T, cmap="gray", origin="lower")
axes[0].add_patch(patches.Rectangle(
    (start[0], start[1]), img_c.shape[0], img_c.shape[1],
    fill=False, edgecolor="yellow", linewidth=1,
))
axes[0].set_title(f"before crop {img_r.shape} + crop box")
axes[1].imshow(img_c.get_fdata()[:, :, z_c].T, cmap="gray", origin="lower")
axes[1].contour((np.asarray(seg_c.dataobj)[:, :, z_c] > 0).T, levels=[0.5], colors="red", linewidths=0.8)
axes[1].set_title(f"after crop {img_c.shape} + mask")
for ax in axes:
    ax.axis("off")
plt.show()

### 6. Pad to a fixed shape

First find the largest shape after resample → crop, over all subjects (slow, run once)

In [ ]:
shapes = []
for img_path, seg_path in pairs:
    img = fill_nan(reorient_to_ras(nib.load(img_path)))
    seg = reorient_to_ras(nib.load(seg_path))
    img, seg = resample_to_spacing(img, seg, spacing=TARGET_SPACING)
    img, seg = crop_to_foreground(img, seg)
    shapes.append(img.shape)

print("max shape per axis:", np.array(shapes).max(axis=0))

In [ ]:
# Round each axis UP to the next multiple of 16 (needed by the U-Net)
max_shape = np.array(shapes).max(axis=0)
TARGET_SHAPE = tuple(int(n) for n in np.ceil(max_shape / 16) * 16)

print("max shape per axis:", max_shape)
print("target shape:     ", TARGET_SHAPE)

In [ ]:
from src.preprocess import pad_to_shape

img_path, seg_path = pairs[0]
img = fill_nan(reorient_to_ras(nib.load(img_path)))
seg = reorient_to_ras(nib.load(seg_path))
img, seg = resample_to_spacing(img, seg, spacing=TARGET_SPACING)
img, seg = crop_to_foreground(img, seg)
img_p, seg_p = pad_to_shape(img, seg, target_shape=TARGET_SHAPE)

print("Before:", img.shape, "After:", img_p.shape, seg_p.shape)
print("Labels unchanged:", np.array_equal(np.unique(np.asarray(seg.dataobj)), np.unique(np.asarray(seg_p.dataobj))))
print("Brain voxels unchanged:", np.count_nonzero(img.get_fdata()) == np.count_nonzero(img_p.get_fdata()))
print("Still RAS:", nib.aff2axcodes(img_p.affine))

# Visual check: middle slice, brain should be centred with black padding around it
z = img_p.shape[2] // 2
plt.imshow(img_p.get_fdata()[:, :, z].T, cmap="gray", origin="lower")
plt.contour((np.asarray(seg_p.dataobj)[:, :, z] > 0).T, levels=[0.5], colors="red", linewidths=0.8)
plt.title(f"padded {img_p.shape} + mask")
plt.axis("off")
plt.show()

### 7. Normalize intensity values

In [ ]:
from src.preprocess import scale_intensity

mins = [nib.load(p).get_fdata(dtype=np.float32).min() for p, _ in pairs]
print("lowest raw intensity across BOBs:", min(mins))

img_path, seg_path = pairs[0]
img = fill_nan(reorient_to_ras(nib.load(img_path)))
img_n = scale_intensity(img)
data = img_n.get_fdata()
print("min / max after:", data.min(), data.max())
print("background still 0:", np.all(data[img.get_fdata() == 0] == 0))

### 8. Remapping labels -- awaiting response